In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "pretraining_results_with_metadata.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all

In [ ]:
METRIC = 'rsa_ve_test'

In [ ]:
subfigsize = (7.5, 7)
subfigsize = (9, 7)
subfigsize = (9, 6)
subfigsize = (8, 5)
nrows, ncols = 1, 1

zoom = 1.0
zoom = 0.75
figsize = (subfigsize[0] * zoom * ncols, subfigsize[1] * zoom * nrows)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
    )

ax = axes

data = df_all.copy()
data = data[(data.is_adversarially_finetuned != True) & (data.is_selfsupervised_learning != True)]
data = data[data.model_id.apply(lambda x: 'flex' not in x.lower())]
# data = data[data.backbone_arch_family == 'ViT']

data = apply_hiearchical_aggregation(
    df=data,
    groupby_cols=GENERIC_GROUPING_COLUMNS,
    agg_cols=METRICS_COMPACT,
    agg_func='mean'
)


data["Architecture Family"] = data['backbone_arch_family']
data['Model Source'] = data['backbone_source']

arch_families = data['Architecture Family'].unique()
n_arch = len(arch_families)


# ---- spvvs: lines + markers ----
sns.lineplot(
    data=data[data['Model Source'] == 'spvvs'],
    x='pretraining_n_samples',
    y=METRIC,
    hue='Architecture Family',
    # marker='o',
    marker=None,
    linestyle='-',
    ax=ax,
    palette=ARCHITECUTURE_FAMILY_COLORS,
    legend=False,   # we will build legends manually
)
sns.scatterplot(
    data=data[data['Model Source'] == 'spvvs'],
    x='pretraining_n_samples',
    y=METRIC,
    hue='Architecture Family',
    marker='o',
    linestyle='-',
    ax=ax,
    palette=ARCHITECUTURE_FAMILY_COLORS,
    alpha=0.3,
    legend=False,   # we will build legends manually
)

# ---- timm: markers only ----
sns.lineplot(
    data=data[data['Model Source'] == 'timm'],
    x='pretraining_n_samples',
    y=METRIC,
    hue='Architecture Family',
    marker=None,
    # linestyle='None',
    linestyle='--',
    # markersize=10,
    errorbar=None,
    ax=ax,
    palette=ARCHITECUTURE_FAMILY_COLORS,
    legend=False,
)
sns.scatterplot(
    data=data[data['Model Source'] == 'timm'],
    x='pretraining_n_samples',
    y=METRIC,
    hue='Architecture Family',
    marker='X',
    # linestyle='None',
    linestyle='--',
    # markersize=10,
    # errorbar=None,
    ax=ax,
    palette=ARCHITECUTURE_FAMILY_COLORS,
    legend=False,
)

arch_families = data['Architecture Family'].unique()

color_handles = [
    Line2D(
        [0], [0],
        color=ARCHITECUTURE_FAMILY_COLORS[arch],
        lw=3,
        label=arch,
    )
    for i, arch in enumerate(arch_families)
]

color_legend = ax.legend(
    handles=color_handles,
    title="Architecture family",
    frameon=True,
    loc="center right",
)

ax.add_artist(color_legend)

source_handles = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='-',
        color='black',
        markersize=8,
        label='spvvs',
    ),
    Line2D(
        [0], [0],
        marker='X',
        linestyle='--',
        color='black',
        markersize=8,
        label='timm',
    ),
]

ax.legend(
    handles=source_handles,
    title="Model source",
    frameon=True,
    loc="lower right",
)


# ax.set_title(benchmark_v, fontsize=16, fontweight='bold')
ax.set_xlabel('Pretraining Samples (D)', fontsize=12, fontweight='bold')
ax.set_ylabel('Alignment Score (S)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Alignment (S)', fontsize=12, fontweight='bold')
ax.set_xscale('log')
# ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.1, 0.2, 0.3, 0.4,  0.5, 0.6])
ax.spines[['right', 'top']].set_visible(False)
    
ax.set_title("Alignment Across Architecture Families", fontsize=16, fontweight='bold')
    
plt.tight_layout()

# figures_dir = save_dir
# fig_name = 'fig3_architecture_families_comparison'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)

figures_dir = save_dir
fig_name = f'metrics-pretraining-{METRIC}-samples-arch_family'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)
